# Campaign · Stage A — cache **Moirai-2** embeddings for C-MAPSS (FD001–FD004)

Stage A of the cross-TSFM C-MAPSS campaign. It embeds FD001–FD004 with the single frozen TSFM **`Salesforce/moirai-2.0-R-small`** and writes the `.npz` embedding caches to Google Drive. **No head training and no baselines run here** — that is Stage B (`score.ipynb`), which reads all five caches from Drive on ONE core runtime.

**Before running:**
1. `Runtime ▸ Change runtime type ▸ GPU`.
2. Use a **fresh runtime, one model per runtime** — the five backbones have mutually incompatible dependencies (see `requirements/README.md`) and must never share an environment. If you ran another model's notebook here, `Runtime ▸ Restart session`.
3. Run the cells top to bottom. The cache writes are **restartable**: re-running resumes and skips datasets already cached.

The canonical `Config` below is **identical** across every Stage-A notebook and `score.ipynb` for the cache-key fields (dataset / window / sensors / max_rul / pooling / context / condition_norm), so Stage B finds exactly the caches Stage A wrote (CHANGES.md §12 winner shape: context 256, mean pooling).

> **Note.** `requirements/moirai.txt` **downgrades** torch to `2.4.1` (uni2ts pins `torch<2.5`) with the matching `torchvision==0.19.1`.

In [ ]:
# 1) clone the campaign branch (shallow) — same pattern as notebooks/verify/*
%cd /content
!git clone --depth 1 --branch claude/c-mapss-colab-campaign-d5yut6 https://github.com/blozanod/Predictive-Maintenance-LSTM.git 2>/dev/null || echo "(already cloned — reusing)"
%cd /content/Predictive-Maintenance-LSTM
import sys; sys.path.insert(0, '/content/Predictive-Maintenance-LSTM')   # import src.* from the fresh clone

In [ ]:
# 2) install ONLY Moirai-2's isolated stack (see requirements/ for the pin rationale).
#    This install CHANGES torch/torchvision. After it finishes, do
#    Runtime ▸ Restart session, then re-run from the Drive-mount cell below
#    (the install is cached, so the restart is quick) before importing src.*.
!pip install -r requirements/moirai.txt

In [ ]:
# 3) mount Google Drive — it holds the persistent embedding cache/ (the Stage A → Stage B hand-off)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 4) the CANONICAL config — identical across every Stage-A notebook and score.ipynb for
#    every cache-key field. Only model_name differs per notebook. head_features is a
#    Stage-B knob and does NOT change the embedding cache (CHANGES.md §9/§12).
from src.config import Config

DRIVE = '/content/drive/MyDrive/pdm_tsfm'          # Drive folder for the shared cache/ (+ results/)
MODEL = 'Salesforce/moirai-2.0-R-small'   # this runtime's ONE model (src/models EMBEDDERS key)

config = Config(
    data_root='Data',                  # C-MAPSS is committed in the repo clone
    cache_dir=f'{DRIVE}/cache',        # embeddings persist here for Stage B
    results_dir=f'{DRIVE}/results',
    model_name=MODEL,
    tsfm_context_length=256,           # recorded FD001 winner (CHANGES.md §12)
    pooling='mean',
    head_features='emb+locscale',      # Stage-B only; harmless here (not a cache key)
    # embed_batch_size=64,             # lower for a T4 (esp. many-channel datasets)
)
config

In [ ]:
# 5) Stage A — embed FD001–FD004 with this ONE model and cache to Drive (restartable).
#    stages=['cache'] does the Stage-A embedding pass only; the embedder auto-detects the
#    GPU. No sweep / baselines here.
import torch
from src.campaign import run_campaign

device = 'cuda' if torch.cuda.is_available() else 'cpu'
summary = run_campaign(
    config,
    models=[MODEL],
    datasets=['FD001', 'FD002', 'FD003', 'FD004'],
    stages=['cache'],
    device=device,
)
summary

## Stage A (horizon half) — embed every test cycle too

Stage B's horizon-stratified + earliness/cost figures need one context per **test cycle** embedded (a sidecar cache, keyed identically to the main cache so it invalidates with it but never touches it — `src/horizon.py`). That embedding is GPU work, so it belongs here on the backbone runtime. `build_horizon_cache` is idempotent and **restartable**; Stage B then finds these caches on Drive and needs no backbone.

In [ ]:
from src.horizon import build_horizon_cache

for ds in ['FD001', 'FD002', 'FD003', 'FD004']:
    p = build_horizon_cache(config.replace(dataset=ds, sensor_columns=None))
    print('horizon cache:', p)